# Feature Engineering

Takes the cleaned + weather-joined dataset and produces a lean, model-ready parquet.

**Input:** `data/processed/ev_cleaned_hourly_weather.parquet` (10.2M rows × 45 cols)  
**Output:** `data/processed/ev_features.parquet`

---

## Feature Summary

### Target
| Column | Description |
|--------|-------------|
| `target` | Hourly charging load (kWh) per station — renamed from `load_kwh_clean` |

### Input Features (33)

| Category | Features | Count |
|----------|----------|-------|
| **Cyclical time** | `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos`, `month_sin`, `month_cos` | 6 |
| **Binary time** | `is_weekend`, `is_holiday` | 2 |
| **Year** | `year` | 1 |
| **Lag** | `load_lag_1h`, `load_lag_2h`, `load_lag_3h`, `load_lag_24h` | 4 |
| **Rolling** | `load_roll_mean_6h/12h/24h/7d`, `load_roll_std_6h/12h/24h/7d` | 8 |
| **Weather** | `temperature_c`, `humidity_pct`, `precipitation_mm`, `wind_speed_kmh`, `cloud_cover_pct`, `is_raining` | 6 |
| **Station (encoded)** | `city_code`, `public_private_code`, `business_type_code`, `contract_type_code` | 4 |
| **Station (numeric)** | `contract_power_kw_norm`, `total_quantity_norm` | 2 |

Plus `customer_id` and `timestamp_hour` kept for bookkeeping (not model input).

### Dropped Columns
| Group | Columns | Reason |
|-------|---------|--------|
| Redundant load | `load_kwh_hourly`, `load_kwh_hourly_scaled` | Duplicate of target |
| Quality flags | `imputed_ffill`, `load_flag_*`, `partial_hour*`, `observed/missing_quarters` | Used for filtering only |
| DR columns | `is_dr_event`, `dr_performance_kwh_hourly`, `exclude_from_baseline` | DR rows removed entirely |
| Raw time ints | `hour`, `day_of_week`, `month`, `season` | Replaced by cyclical encodings |
| Strings | `address`, `detailed_type`, `participate_program`, `province` | High cardinality / single value |
| Raw categoricals | `city`, `public_private`, `business_type`, `contract_type` | Replaced by `_code` versions |
| Raw station nums | `contract_power_kw`, `total_quantity` | Replaced by `_norm` versions |
| Charger breakdown | `charger_7kw/8kw`, `other_slow/fast_charger`, `charger_50kw` | Redundant with `total_quantity_norm` |

### Train/Test Split
- **Train:** 2021 (temporal split, no data leakage)
- **Test:** 2022

## 0. Setup & Load

In [ ]:
import warnings
import pandas as pd
import numpy as np
from pathlib import Path

warnings.filterwarnings('ignore')

# Find project root
current = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [current, *current.parents]:
    if (candidate / 'data' / 'raw').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project root containing data/raw')

IN_PATH  = PROJECT_ROOT / 'data' / 'processed' / 'ev_cleaned_hourly_weather.parquet'
OUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ev_features.parquet'

df = pd.read_parquet(IN_PATH)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols')
print(f'Columns: {df.columns.tolist()}')

## 1. Baseline Filtering

- Drop rows where `load_kwh_clean` is NaN (28.3% of dataset — genuinely missing, not usable for training)
- Exclude DR event hours from training data (different operating regime)
- Handle outliers: keep them but cap extreme values at the 99.5th percentile per station

In [ ]:
n_before = len(df)

# Drop rows with no load measurement
df = df.dropna(subset=['load_kwh_clean'])
print(f'Dropped NaN load: {n_before:,} → {len(df):,} ({n_before - len(df):,} removed)')

# Exclude DR event hours
n_dr = df['is_dr_event'].sum()
df = df[~df['is_dr_event']].copy()
print(f'Dropped DR hours: {n_dr:,} removed → {len(df):,} remaining')

# Cap extreme outliers per station at 99.5th percentile
caps = df.groupby('customer_id')['load_kwh_clean'].quantile(0.995)
df = df.merge(caps.rename('load_cap'), on='customer_id', how='left')
n_capped = (df['load_kwh_clean'] > df['load_cap']).sum()
df['load_kwh_clean'] = df['load_kwh_clean'].clip(upper=df['load_cap'])
df = df.drop(columns='load_cap')
print(f'Capped outliers: {n_capped:,} values clipped to per-station 99.5th percentile')

print(f'\nFiltered dataset: {len(df):,} rows')

## 2. Cyclical Time Encoding

Encode `hour`, `day_of_week`, and `month` as sin/cos pairs so the model understands  
that hour 23 and hour 0 are adjacent, December and January are adjacent, etc.

In [ ]:
def cyclical_encode(series, period):
    """Encode a numeric series as sin/cos pair with given period."""
    angle = 2 * np.pi * series / period
    return np.sin(angle).astype('float32'), np.cos(angle).astype('float32')

# Hour of day (period = 24)
df['hour_sin'], df['hour_cos'] = cyclical_encode(df['hour'], 24)

# Day of week (period = 7)
df['dow_sin'], df['dow_cos'] = cyclical_encode(df['day_of_week'], 7)

# Month (period = 12)
df['month_sin'], df['month_cos'] = cyclical_encode(df['month'], 12)

print('Added: hour_sin, hour_cos, dow_sin, dow_cos, month_sin, month_cos')
print(f'Sample (hour=0):  sin={df.loc[df["hour"]==0, "hour_sin"].iloc[0]:.3f}, cos={df.loc[df["hour"]==0, "hour_cos"].iloc[0]:.3f}')
print(f'Sample (hour=12): sin={df.loc[df["hour"]==12, "hour_sin"].iloc[0]:.3f}, cos={df.loc[df["hour"]==12, "hour_cos"].iloc[0]:.3f}')

## 3. Lag & Rolling Features

Per-station temporal features. These require the data to be sorted by `(customer_id, timestamp_hour)`.  
Rows at the start of each station's history will have NaN lags — these are dropped at the end.

In [ ]:
df = df.sort_values(['customer_id', 'timestamp_hour']).reset_index(drop=True)

grouped = df.groupby('customer_id')['load_kwh_clean']

# --- Lag features ---
for lag in [1, 2, 3, 24]:
    df[f'load_lag_{lag}h'] = grouped.shift(lag).astype('float32')

print('Added lag features: load_lag_1h, load_lag_2h, load_lag_3h, load_lag_24h')

# --- Rolling features ---
for window in [6, 12, 24, 168]:  # 168 = 7 days
    label = f'{window}h' if window < 168 else '7d'
    rolling = grouped.transform(lambda x: x.rolling(window, min_periods=1).mean())
    df[f'load_roll_mean_{label}'] = rolling.astype('float32')
    rolling_std = grouped.transform(lambda x: x.rolling(window, min_periods=1).std())
    df[f'load_roll_std_{label}'] = rolling_std.astype('float32')

print('Added rolling features: mean & std for 6h, 12h, 24h, 7d windows')
print(f'NaN count from lags (will be dropped later): {df["load_lag_24h"].isna().sum():,}')

## 4. Weather Features\n\nThe raw weather columns (temperature, humidity, precipitation, wind speed, cloud cover, is_raining) are already joined from notebook 02 — Weather Data. No derived features needed; tree-based models learn thresholds on their own.

In [ ]:
weather_cols = ['temperature_c', 'humidity_pct', 'precipitation_mm',
                'wind_speed_kmh', 'cloud_cover_pct', 'is_raining']

print('Weather columns (kept as-is from 02 - Weather Data):')
for col in weather_cols:
    print(f'  {col:<20} dtype={df[col].dtype}  nulls={df[col].isna().sum()}')

## 5. Station Features

Encode categorical station attributes for the model.

In [ ]:
# --- Label encode low-cardinality categoricals ---
cat_cols = ['city', 'public_private', 'business_type', 'contract_type']

for col in cat_cols:
    df[col] = df[col].astype('category')
    df[f'{col}_code'] = df[col].cat.codes.astype('int8')
    mapping = dict(enumerate(df[col].cat.categories))
    print(f'{col}: {mapping}')

# --- Normalize numeric station attributes ---
from sklearn.preprocessing import MinMaxScaler

num_station_cols = ['contract_power_kw', 'total_quantity']
scaler = MinMaxScaler()
df[[f'{c}_norm' for c in num_station_cols]] = scaler.fit_transform(
    df[num_station_cols].astype('float32')
).astype('float32')

print(f'\nNormalized: {num_station_cols} → [0, 1]')

## 6. Target Variable & Train/Test Split Strategy

- **Target:** `load_kwh_clean` — hourly charging load per station
- **Split:** Temporal — train on 2021, test on 2022 (no data leakage across time)

In [ ]:
df = df.rename(columns={'load_kwh_clean': 'target'})

train_mask = df['year'] == 2021
test_mask  = df['year'] == 2022

print(f'Target: "target" (renamed from load_kwh_clean)')
print(f'Train (2021): {train_mask.sum():,} rows')
print(f'Test  (2022): {test_mask.sum():,} rows')
print(f'Ratio: {train_mask.sum() / len(df) * 100:.1f}% / {test_mask.sum() / len(df) * 100:.1f}%')

## 7. Drop Unused Columns

Remove columns the model shouldn't see: identifiers, raw strings, quality flags,  
redundant load variants, and raw time integers (replaced by cyclical encodings).

In [ ]:
drop_cols = [
    # --- Identifiers & strings (not model inputs) ---
    'address', 'province', 'detailed_type', 'participate_program',

    # --- Raw categoricals (replaced by _code versions) ---
    'city', 'public_private', 'business_type', 'contract_type',

    # --- Redundant load columns ---
    'load_kwh_hourly', 'load_kwh_hourly_scaled',

    # --- Quality flags (used for filtering, not model features) ---
    'imputed_ffill', 'load_flag_negative_corrected',
    'load_flag_outlier', 'load_flag_outlier_zscore', 'load_flag_outlier_iqr',
    'partial_hour', 'partial_hour_scaled',
    'observed_quarters', 'missing_quarters',

    # --- DR columns (already filtered out DR rows) ---
    'is_dr_event', 'dr_performance_kwh_hourly', 'exclude_from_baseline',

    # --- Raw time integers (replaced by cyclical sin/cos) ---
    'hour', 'day_of_week', 'month',

    # --- Season string (redundant with month encoding) ---
    'season',

    # --- Raw station numerics (replaced by _norm versions) ---
    'contract_power_kw', 'total_quantity',

    # --- Charger breakdown (too granular, total_quantity_norm captures it) ---
    'charger_7kw', 'charger_8kw', 'other_slow_charger',
    'charger_50kw', 'other_fast_charger',
]

# Only drop columns that actually exist
drop_existing = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_existing)

print(f'Dropped {len(drop_existing)} columns')
print(f'Remaining: {df.shape[1]} columns')
print(f'\nFinal columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col:<25} {df[col].dtype}')

## 8. Drop NaN Rows from Lag Features

Lag/rolling features create NaN at the start of each station's history. Drop these.

In [ ]:
n_before = len(df)
df = df.dropna().reset_index(drop=True)
print(f'Dropped NaN from lags: {n_before:,} → {len(df):,} ({n_before - len(df):,} removed)')

## 9. Save

In [ ]:
df.to_parquet(OUT_PATH, index=False)

size_mb = OUT_PATH.stat().st_size / 1e6
print(f'Saved → {OUT_PATH}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} cols')
print(f'Size: {size_mb:.1f} MB')
print(f'\nFinal column list:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col:<25} {df[col].dtype}')